In [1]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

/Users/konszym/Desktop/RÓŻNE/VS CODE/hacknarok-fiatpandas/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 0 if torch.backends.mps.is_available() else -1

In [3]:
classifier = pipeline(
    "text-classification", 
    model="bhadresh-savani/distilbert-base-uncased-emotion", 
    top_k=None, # Zwróć wyniki dla wszystkich 6 emocji, a nie tylko dla tej najsilniejszej
    device=device 
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11723.81it/s]


In [4]:
def get_emotion_vector(text):
    clean_text = str(text)[:512]
    results = classifier(clean_text)
    return [res['score'] for res in results[0]]

In [6]:
df = pd.read_csv("../../data/cleaned_data.csv")
features = []
for text in tqdm(df['cleaned_statement']):
    features.append(get_emotion_vector(text))

100%|██████████| 29995/29995 [06:43<00:00, 74.38it/s]


In [7]:
emotion_cols = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
features_df = pd.DataFrame(features, columns=emotion_cols)
features_df['label'] = df['label']

In [8]:
features_df.to_csv('../../data/features_for_random_forest.csv', index=False)